In [34]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = 'meta-llama/Llama-3.2-1B'
base_model = AutoModelForCausalLM.from_pretrained(model_id)

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 2696.02it/s, Materializing param=model.norm.weight]                              


In [37]:
import torch.nn as nn

def get_base_model_parameter(model, rank, alpha):
    base_model_parameters = []
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            base_model_parameters.append(
              [module.in_features, module.out_features, rank, alpha]
            )
        else:
            base_model_parameters.extend(get_base_model_parameter(module, rank, alpha))
    
    return base_model_parameters

base_model_parameters = get_base_model_parameter(base_model, rank=16, alpha=16)


In [39]:
base_model_parameters[0]

[2048, 2048, 16, 16]

In [38]:
len(base_model_parameters)

113

In [ ]:
import torch
import math
from huggingface_hub import PyTorchModelHubMixin

class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank, dtype=torch.bfloat16))
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim, dtype=torch.bfloat16))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

class LlamaSummarizationLoRALayers(nn.Module, PyTorchModelHubMixin):
    def __init__(self):
        super().__init__()
        self.lora_layers = nn.ModuleList(
            [
                LoRALayer(base_model_parameter[0], base_model_parameter[1], base_model_parameter[2], base_model_parameter[3]) 
                for base_model_parameter in base_model_parameters
            ]
        )

In [43]:
model_id = 'SauravP97/llama-3.2-1B-summarization-loraa-layers'
llama_summarizer_lora_layers = LlamaSummarizationLoRALayers.from_pretrained(model_id)

In [44]:
llama_summarizer_lora_layers

LlamaSummarizationLoRALayers(
  (lora_layers): ModuleList(
    (0-112): 113 x LoRALayer()
  )
)

In [45]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, lora_layer):
        super().__init__()
        self.linear = linear
        self.lora = lora_layer

    def forward(self, x):
        return self.linear(x) + self.lora(x)

In [46]:
def replace_linear_with_preloaded_lora(model, lora_layers, cur_index):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            print(f'Initialized with Pre-loaded LoRA Layer: {cur_index}')
            setattr(model, name, LinearWithLoRA(module, lora_layers[cur_index]))
            cur_index = cur_index + 1
        else:
            cur_index = replace_linear_with_preloaded_lora(module, lora_layers, cur_index)
    
    return cur_index

In [47]:
replace_linear_with_preloaded_lora(base_model, llama_summarizer_lora_layers.lora_layers, 0)

Initialized with Pre-loaded LoRA Layer: 0
Initialized with Pre-loaded LoRA Layer: 1
Initialized with Pre-loaded LoRA Layer: 2
Initialized with Pre-loaded LoRA Layer: 3
Initialized with Pre-loaded LoRA Layer: 4
Initialized with Pre-loaded LoRA Layer: 5
Initialized with Pre-loaded LoRA Layer: 6
Initialized with Pre-loaded LoRA Layer: 7
Initialized with Pre-loaded LoRA Layer: 8
Initialized with Pre-loaded LoRA Layer: 9
Initialized with Pre-loaded LoRA Layer: 10
Initialized with Pre-loaded LoRA Layer: 11
Initialized with Pre-loaded LoRA Layer: 12
Initialized with Pre-loaded LoRA Layer: 13
Initialized with Pre-loaded LoRA Layer: 14
Initialized with Pre-loaded LoRA Layer: 15
Initialized with Pre-loaded LoRA Layer: 16
Initialized with Pre-loaded LoRA Layer: 17
Initialized with Pre-loaded LoRA Layer: 18
Initialized with Pre-loaded LoRA Layer: 19
Initialized with Pre-loaded LoRA Layer: 20
Initialized with Pre-loaded LoRA Layer: 21
Initialized with Pre-loaded LoRA Layer: 22
Initialized with Pre-

113

In [48]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=2048, bias=False)
            (lora): LoRALayer()
          )
          (k_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=512, bias=False)
            (lora): LoRALayer()
          )
          (v_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=512, bias=False)
            (lora): LoRALayer()
          )
          (o_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=2048, bias=False)
            (lora): LoRALayer()
          )
        )
        (mlp): LlamaMLP(
          (gate_proj): LinearWithLoRA(
            (linear): Linear(in_features=2048, out_features=8192, bias=False)
            (lora): LoRALayer()


In [49]:
model_id = 'meta-llama/Llama-3.2-1B'
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [50]:
tokenizer.pad_token_id = 1131
tokenizer.pad_token

'...'

In [55]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [51]:
import torch

def generate_content(prompt, model):
    encoded_ids = tokenizer.encode(prompt)
    encoded_ids = torch.tensor(encoded_ids, dtype=torch.int64, device=device)

    encoded_ids = encoded_ids.view(1, encoded_ids.shape[-1])

    generated_ids = model.generate(
        encoded_ids, max_length=500, pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(generated_ids)[0]

In [52]:
prompt = '''<|begin_of_text|> Once upon a time, there was a beautiful cat. The cat had a big jar of ink. One day, the cat wanted to make a pretty picture with the ink. The cat dipped its tail in the ink and started to draw on a big white paper. As the cat was drawing, a little mouse came to watch. The mouse wanted to help the cat, but the cat did not see the mouse. The mouse stepped on the ink and then walked on the paper. The cat saw the messy footprints and felt embarrass. Then, something unexpected happened. The footprints made a new picture on the paper. The cat and the mouse looked at it and saw that it was a beautiful heart. They both laughed and became friends. From that day, they made many beautiful pictures together. 
Summary: 
'''

In [53]:
prompt2 = '''<|begin_of_text|> Once upon a time, there was a little bird. The bird was white and had a pretty song. One day, the bird saw a big pit. The pit was deep and dark. The bird wanted to see what was inside, so it flew down to take a look. As the bird got closer, the pit began to rise up and swallow the bird. The bird tried to fly away, but it was too late. The pit had taken the bird down. The moral of the story is that we should be careful when we see something dangerous. We should not go near it, even if we are curious. It is better to be safe than sorry. 
Summary: 
'''

In [57]:
print(generate_content(prompt2, base_model))

<|begin_of_text|><|begin_of_text|> Once upon a time, there was a little bird. The bird was white and had a pretty song. One day, the bird saw a big pit. The pit was deep and dark. The bird wanted to see what was inside, so it flew down to take a look. As the bird got closer, the pit began to rise up and swallow the bird. The bird tried to fly away, but it was too late. The pit had taken the bird down. The moral of the story is that we should be careful when we see something dangerous. We should not go near it, even if we are curious. It is better to be safe than sorry. 
Summary: 
A curious little bird flies down to a deep pit and gets swallowed by it, teaching the moral to be careful when exploring dangerous things. <|end_of_text|>
